# XANES → oxidation state — end-to-end walkthrough

This notebook reproduces the headline result for Mn end-to-end:
fetch → clean → split → train ensemble → temperature-scale → evaluate
→ headline figure. Other elements follow the same recipe via the CLI.

Requires `MP_API_KEY` in the environment to fetch fresh data; if the
cached parquet already exists under `data/processed/`, fetching is
skipped.


In [ ]:
import os
from pathlib import Path

from xanes_oxstate.data.run import build_element_dataset
from xanes_oxstate.eval.run import evaluate_element

ELEMENT = "Mn"
ROOT = Path.cwd().parent

paths = build_element_dataset(
    ELEMENT,
    raw_dir=ROOT / "data" / "raw",
    processed_dir=ROOT / "data" / "processed",
    report_dir=ROOT / "data" / "reports",
)
paths


In [ ]:
metrics = evaluate_element(
    ELEMENT,
    processed_dir=ROOT / "data" / "processed",
    ckpt_dir=ROOT / "checkpoints",
    metrics_dir=ROOT / "metrics",
    epochs=50,
)
metrics["accuracy"]


In [ ]:
import json
import numpy as np
from xanes_oxstate.eval.plots import (
    per_element_accuracy_bar,
    confusion_small_multiples,
)

acc = metrics["accuracy"]
labels = [metrics["class_to_ox_state"][str(i)]
          for i in range(len(metrics["class_to_ox_state"]))]
cm = np.load(ROOT / "metrics" / f"{ELEMENT}_cm.npy")

per_element_accuracy_bar({ELEMENT: acc})
confusion_small_multiples({ELEMENT: (cm, labels)}, n_cols=1)


## Physics findings

Use `xanes_oxstate.physics.analysis` to inspect each entry in
`metrics["confused_pairs"]` — overlay mean spectra, fit a pairwise
logistic regression to find the discriminative spectral region, and
group failures by coordination number to look for structural patterns.
Write up the result in `docs/physics_findings.md`.
